# RGB (VisDrone DET) - 02 Entrenar

Que hace: entrena un YOLO sobre VisDrone y guarda el run (pesos, metricas, plots) directo en Drive.

Que NO hace: no evalua en test ni procesa video (eso es `03_testing`).

In [ ]:
import os; os.cpu_count()

## Parametros

Prueba barata antes de un entrenamiento largo: `EPOCHS = 1` y `TRAIN_ARGS["fraction"] = 0.05`.
Si Colab se corto a mitad: poner en `RESUME_RUN` el nombre de la carpeta del run y correr todo de nuevo.

In [ ]:
DATASET_ZIP = "MyDrive/deteccion_drones/datasets/visdrone.zip"
DATASET_DIR = "visdrone"                        # subcarpeta en env.DATA_ROOT
DATA_YAML   = "VisDrone_Dataset/visdrone.yaml"  # relativo a DATASET_DIR
OUT_DIR     = "MyDrive/deteccion_drones/runs/rgb"

MODEL = "yolo26m.pt"          # o "yolo11s.pt" para la comparacion
EPOCHS, IMGSZ, BATCH = 60, 1024, 16    # Despues probar 1280
TAG = ""                      

CLASS_MAP = {"pedestrian": "person", "people": "person"}   # None = clases originales
REMAP_DIR = "visdrone_person"   # subcarpeta en env.DATA_ROOT; si cambias CLASS_MAP, cambia este nombre

TRAIN_ARGS = dict(cos_lr=True, close_mosaic=15, patience=15, cache="ram")   


RESUME_RUN = None#"visdrone_person_yolo26s_1024_20260929-1747"             # nombre de un run cortado para retomarlo, ej "visdrone_yolo11s_1024_20260928-1530"


## Setup

Identica en todos los notebooks. El repo tiene que ser publico y `BRANCH` tiene que estar pusheada.

In [ ]:
REPO = "https://github.com/sbstn-sss/deteccion_drones.git"
REPO_DIR = "/content/deteccion_drones"
BRANCH = "feat/framework"   # cambiar para probar una rama antes del merge, ej "feat/framework"

import sys
if "google.colab" in sys.modules:
    !git -C {REPO_DIR} checkout -q {BRANCH} 2>/dev/null && git -C {REPO_DIR} pull -q 2>/dev/null || git clone -q -b {BRANCH} {REPO} {REPO_DIR}
    !pip install -q -e {REPO_DIR}
    sys.path.insert(0, f"{REPO_DIR}/src")   # pip -e usa un .pth que el kernel ya iniciado no lee: sin esto falla el import

try:  # el IPython de Colab con Python 3.13 trae un autoreload que importa 'imp' (eliminado en 3.12)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload no disponible: reinicia el kernel tras editar src/")
from deteccion import env, data, viz
from deteccion import ExperimentConfig, Experiment, list_runs
env.mount_drive()


## 1. Obtener el dataset

Igual que en `01_explorar`: si el dataset ya esta descomprimido en esta sesion (marcador `.done`), no hace nada.

In [ ]:
dataset_dir = env.DATA_ROOT / DATASET_DIR
env.unzip_once(env.drive_path(DATASET_ZIP), dataset_dir)
data_yaml = env.fix_data_yaml(dataset_dir / DATA_YAML)
if CLASS_MAP:
    data_yaml = data.remap_dataset(data_yaml, CLASS_MAP, env.DATA_ROOT / REMAP_DIR)
print(data_yaml.read_text())   # confirmar: names debe empezar con 0: person



## 2. Configurar el experimento

El nombre del run se arma solo (dataset, modelo, imgsz, tag, fecha): no hay que adivinar sufijos `-2`, `-8`.
La config queda guardada en `experiment.json` dentro del run.

In [ ]:
cfg = ExperimentConfig(data_yaml=str(data_yaml), out_dir=OUT_DIR, model=MODEL, epochs=EPOCHS,
                       imgsz=IMGSZ, batch=BATCH, tag=TAG, train_args=TRAIN_ARGS)
if RESUME_RUN is None:
    exp = Experiment(cfg)
else:
    exp = Experiment.load(env.drive_path(OUT_DIR) / RESUME_RUN, data_yaml=str(data_yaml))
print(exp.run_dir)


## 3. Entrenar

Se escribe en Drive en cada epoca, asi que una desconexion no pierde el run: se retoma con `RESUME_RUN`.

In [ ]:
results = exp.train() if RESUME_RUN is None else exp.resume()


In [ ]:
# ver evolucion

import pandas as pd
df = pd.read_csv(exp.run_dir / "results.csv"); df.columns = df.columns.str.strip()
df.plot(x="epoch", y=["metrics/mAP50-95(B)", "metrics/recall(B)", "metrics/precision(B)"], grid=True)
df.plot(x="epoch", y=["train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss"], grid=True)


In [ ]:
# Ejecutar post cortar el entrenamiento para poder tener los graficos finales

MODEL = f"/content/drive/MyDrive/deteccion_drones/runs/rgb/{RESUME_RUN}/weights/best.pt"
EPOCHS = 15
TAG = "finetune"
TRAIN_ARGS = dict(
    optimizer="SGD", lr0=0.001,   # lr bajo; con optimizer="auto" se ignoraria lr0
    close_mosaic=15,              # = EPOCHS: todo el run sin mosaic
    cos_lr=True, patience=10, workers=2,
)

## 4. Curvas y validacion

Los plots son los que Ultralytics ya genero en el run. Las metricas se miran en **val**; test se usa una sola vez al final, en `03_testing`.

In [ ]:
viz.show_run_plots(exp.run_dir)


In [ ]:
exp.val("val")


## Notas

_(completar: que se cambio respecto al run anterior y que resultado dio)_